# CRF POS Tagging

In [ ]:
# ============================================================
# 07 - POS TAGGING USING CRF
# ============================================================

# Install once:
# !pip install sklearn-crfsuite

import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.metrics import classification_report

# ============================================================
# SMALL TRAINING CORPUS
# ============================================================

train_sents = [
    [
        ("tiger", "NN"),
        ("chases", "VB"),
        ("deer", "NN")
    ],
    [
        ("civet", "NN"),
        ("chases", "VB"),
        ("deer", "NN")
    ],
    [
        ("civet", "NN"),
        ("makes", "VB"),
        ("loud", "RB")
    ],
    [
        ("dog", "NN"),
        ("runs", "VB"),
        ("quickly", "RB")
    ],
]

# ============================================================
# FEATURE EXTRACTION
# ============================================================

def word2features(sent, i):
    word = sent[i][0]
    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word[:3]": word[:3],
        "word[:2]": word[:2],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
        "position": i,
        "is_first": i == 0,
        "is_last": i == len(sent) - 1,
    }

    if i > 0:
        previous_word = sent[i - 1][0]
        features.update({
            "-1:word.lower()": previous_word.lower(),
            "-1:word[-3:]": previous_word[-3:],
        })
    else:
        features["BOS"] = True

    if i < len(sent) - 1:
        next_word = sent[i + 1][0]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word[-3:]": next_word[-3:],
        })
    else:
        features["EOS"] = True

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [tag for word, tag in sent]

def sent2tokens(sent):
    return [word for word, tag in sent]

# Convert training data
X_train = [sent2features(sent) for sent in train_sents]
y_train = [sent2labels(sent) for sent in train_sents]

# ============================================================
# TRAIN CRF
# ============================================================

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

crf.fit(X_train, y_train)

# ============================================================
# PREDICT A NEW SENTENCE
# ============================================================

def crf_tag_sentence(words):
    # CRF feature extraction expects word/tag pairs,
    # but tags are placeholders because this is test input.
    sent = [(word, "NN") for word in words]

    features = sent2features(sent)
    predicted_tags = crf.predict_single(features)

    return list(zip(words, predicted_tags))

test_sentence = ["tiger", "chases", "deer"]

result = crf_tag_sentence(test_sentence)

print("Input:")
print(test_sentence)

print("\nCRF output:")
for word, tag in result:
    print(word, "/", tag)

# ============================================================
# EVALUATION
# ============================================================

y_pred = crf.predict(X_train)

print("\nClassification report:")
print(
    classification_report(
        [tag for sent in y_train for tag in sent],
        [tag for sent in y_pred for tag in sent],
        zero_division=0
    )
)

print("\nCRF flat accuracy:")
print(metrics.flat_accuracy_score(y_train, y_pred))

print("\nCRF flat F1:")
print(metrics.flat_f1_score(y_train, y_pred, average="weighted"))

# ============================================================
# ============================================================
# train_sents = [...]
#
# Test sentence:
# test_sentence = [...]
#
# IMPORTANT:
# For each training sentence, the number of words must equal
# the number of POS labels.
#
# Example:
# [("dog","NN"), ("runs","VB")]
# has 2 words and 2 labels.


Input:
['tiger', 'chases', 'deer']

CRF output:
tiger / NN
chases / VB
deer / NN

Classification report:
              precision    recall  f1-score   support

          NN       1.00      1.00      1.00         7
          RB       1.00      1.00      1.00         2
          VB       1.00      1.00      1.00         4

    accuracy                           1.00        13
   macro avg       1.00      1.00      1.00        13
weighted avg       1.00      1.00      1.00        13

CRF flat accuracy:
1.0

CRF flat F1:
1.0
